# **Four maps on one input: attention, attention × gradient, rollout and Chefer**

A practice for the module ["Attention-based explainability"](https://ai-interpretability.school).

The module covers four ways to build an importance map from a transformer's attention: the raw
attention map of one layer, attention × gradient, attention rollout and the method of Chefer et al.
Here all four are computed on **the same input** — a picture with a cat and a dog — with the same
ViT-Small model, and compared by number:

1. whether the map depends on which class we ask about;
2. how similar the maps of different methods are to one another;
3. what share of each map's mass falls into the cat box and into the dog box.

At the end: the attention sink in GPT-2 small, that is, where most of a language model's attention goes.

The input and the model are the same as in the notebook ["Attribution for transformers"](https://colab.research.google.com/github/SadSabrina/open-xai-materials/blob/main/homeworks/PRACTICE_transformer_attribution_en.ipynb)
of the module "Explanations from axioms": rollout and Chefer are covered there in detail, and here the
raw attention map and attention × gradient are added to them. It takes about two minutes on a CPU.

In [ ]:
!pip install timm -q     # timm is not always installed in Colab

In [ ]:
import io
import types
import urllib.request

import timm
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms

torch.manual_seed(0)
DATA = 'https://raw.githubusercontent.com/SadSabrina/open-xai-materials/main'

model = timm.create_model('vit_small_patch16_224', pretrained=True).eval()
for p in model.parameters():
    p.requires_grad_(False)


def attn_forward(self, x, **kw):
    """Attention that keeps the matrix and the gradient with respect to it: the maps need both."""
    B, N, C = x.shape
    qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
    q, k, v = qkv.unbind(0)
    q, k = self.q_norm(q), self.k_norm(k)
    attn = ((q * self.scale) @ k.transpose(-2, -1)).softmax(dim=-1)
    self.attn_map = attn          # all four maps need the attention matrix
    attn.retain_grad()            # attention × gradient and Chefer need the gradient with respect to it
    return self.proj((attn @ v).transpose(1, 2).reshape(B, N, C))


for block in model.blocks:
    block.attn.fused_attn = False     # fast attention does not return the matrix, so we turn it off
    block.attn.forward = types.MethodType(attn_forward, block.attn)

cfg = model.pretrained_cfg
tf = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(),
                         transforms.Normalize(cfg['mean'], cfg['std'])])


def fetch(path):
    return urllib.request.urlopen(f'{DATA}/' + path, timeout=30).read()


img = tf(Image.open(io.BytesIO(fetch('data/cat_and_dog.jpg'))).convert('RGB'))
img = img.unsqueeze(0).requires_grad_(True)
names = [s.strip() for s in fetch('data/imagenet_classes.txt').decode().split('\n')]
GRID = int(model.patch_embed.num_patches ** 0.5)
print(f'blocks: {len(model.blocks)}, patch grid: {GRID}x{GRID}')

## 1. Attention and its gradient

The library computes attention with a fast kernel that does not return the matrix itself, so fast
attention is turned off and the plain implementation is put in its place. `attn.retain_grad()` keeps
the gradient of the logit with respect to the attention matrix: without it there is nothing to compute
attention × gradient and Chefer from.

One pass `run(cls)` returns the attention matrices of all 12 blocks and the gradients of the logit of
class `cls` with respect to them.

In [ ]:
def run(cls):
    """One pass: attention matrices of all blocks and the gradients of the logit of class cls with respect to them."""
    model.zero_grad()
    img.grad = None
    out = model(img)
    out[0, cls].backward()
    return [b.attn.attn_map.detach()[0] for b in model.blocks], \
        [b.attn.attn_map.grad[0] for b in model.blocks]


CAT, DOG = 281, 243        # tabby and bull mastiff: a cat and a dog
mats, grads = run(DOG)
print(f'dog class: {names[DOG]}')
print(f'attention matrix shape (heads, tokens, tokens): {tuple(mats[0].shape)}')

## 2. Four methods

All four take the row of the `CLS` token — how much it takes from each patch — and lay it back out
into the $14\times14$ grid.

- **attention** is the raw map: the attention of the last block averaged over heads.
- **attention × gradient** is the same matrix multiplied elementwise by the gradient of the class
  logit with respect to it; the positive part is taken and averaged over heads. This answers the main
  objection of Jain & Wallace: an attention weight by itself does not know whether it influenced the
  answer, and the gradient does.
- **rollout** is the attention of all blocks averaged over heads, with an identity matrix added for
  the skip connection, multiplied across layers.
- **Chefer** is the generic variant of Chefer et al.: the same roll-out as rollout, but before averaging
  each matrix is multiplied by the gradient of the logit with respect to it. The full variant from the
  paper adds LRP rules for skip connections and for the product $A\cdot V$; class specificity comes
  from the gradient factor, and it is here.

In [ ]:
def raw(mats, grads):
    """Raw map: attention of the last block, mean over heads."""
    return mats[-1].mean(0)


def attn_x_grad(mats, grads):
    """Attention × gradient: attention of the last block times the logit gradient, positive part, mean over heads."""
    return (grads[-1] * mats[-1]).clamp(min=0).mean(0)


def rollout(mats, grads):
    """Attention rollout: mean over heads, half of the weight on the skip connection, product across layers."""
    n = mats[0].shape[-1]
    r = torch.eye(n)
    for A in mats:
        r = (0.5 * A.mean(0) + 0.5 * torch.eye(n)) @ r
    return r


def chefer(mats, grads):
    """Chefer et al. (generic): the same roll-out, but each matrix is weighted by the gradient of the class logit."""
    n = mats[0].shape[-1]
    r = torch.eye(n)
    for A, G in zip(mats, grads):
        a = (G * A).clamp(min=0).mean(0) + torch.eye(n)
        a = a / a.sum(dim=-1, keepdim=True)
        r = a @ r
    return r


METHODS = {'attention': raw, 'attention x gradient': attn_x_grad, 'rollout': rollout, 'Chefer': chefer}


def to_map(r):
    """The CLS row without CLS itself, laid back out into the patch grid and normalized to [0, 1]."""
    m = r[0, 1:].reshape(GRID, GRID)
    return (m - m.min()) / (m.max() - m.min() + 1e-9)


def corr(a, b):
    a, b = a - a.mean(), b - b.mean()
    return float((a * b).sum() / (a.norm() * b.norm() + 1e-9))

## 3. Does the map depend on the question

We build two maps with each method — asking about the cat and about the dog — and compute the Pearson
correlation between them. First for the two methods without a gradient.

In [ ]:
maps = {}
for cls, label in ((CAT, 'cat'), (DOG, 'dog')):
    m, g = run(cls)
    maps[label] = {name: to_map(f(m, g)) for name, f in METHODS.items()}

for name in ('attention', 'rollout'):
    a, b = maps['cat'][name], maps['dog'][name]
    print(f'{name:22} correlation of the cat and dog maps: {corr(a, b):.6f}   maximum difference: {(a - b).abs().max():.2e}')

**For the raw attention map and for rollout the correlation is exactly one and the difference is
exactly zero.** The attention matrix is computed on the forward pass and knows nothing about the
class, so neither it nor the product of the matrices depends on the question.

**Task 1.** Compute the same for attention × gradient and for Chefer. Which of the two maps changes
more when the question changes from the dog to the cat? The answer is the correlation of the
attention × gradient maps for the cat and the dog, to two decimal places.

In [ ]:
# Your code here

## 4. Are the maps of different methods alike

Pearson correlation between the maps of the four methods for the same question, about the dog.

In [ ]:
# correlation of the maps of the four methods, question about the dog
print(f'{"":22}' + ''.join(f'{n:>22}' for n in METHODS))
for a in METHODS:
    print(f'{a:22}' + ''.join(f'{corr(maps["dog"][a], maps["dog"][b]):22.3f}' for b in METHODS))

The table is symmetric, with ones on the diagonal. In the `Chefer` row, find the method whose map is
closest to it and the method whose map is farthest.

## 5. Where exactly the maps look

Correlation says the maps are different but does not say which of them looks where it should. Let us
mark the boxes of the cat and the dog and see what share of each map's mass falls into each box.

In [ ]:
# The boxes are marked by eye on the 224 by 224 picture, the same as in the notebook "Attribution for transformers"
BOXES = {'cat': (10, 40, 110, 200), 'dog': (115, 30, 215, 205)}


def mass_in(m, box):
    """Share of the map mass that falls inside the box."""
    big = F.interpolate(m[None, None], (224, 224), mode='bilinear', align_corners=False)[0, 0]
    x0, y0, x1, y1 = box
    return float(big[y0:y1, x0:x1].sum() / big.sum())


print(f'{"":22}{"asked":10}{"in cat box":>16}{"in dog box":>17}')
for label in ('cat', 'dog'):
    for name in METHODS:
        m = maps[label][name]
        print(f'{name:22}{label:10}{mass_in(m, BOXES["cat"]) * 100:15.1f}%'
              f'{mass_in(m, BOXES["dog"]) * 100:16.1f}%')

**The rows of the raw attention map and of rollout for the cat and for the dog coincide**: these maps
answer the question "where does the model look at all". **The rows of attention × gradient and Chefer
differ**: ask about the cat, and more mass lands in the cat box.

**Task 2.** The raw attention map puts more mass into the dog box than rollout does, also when we ask
about the cat. Does that mean the model "looks at the dog" when it decides whether there is a cat in the
picture? Base your answer on the numbers in the table.

## 6. Look with your own eyes

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for row, label in enumerate(('cat', 'dog')):
    for col, name in enumerate(METHODS):
        axes[row, col].imshow(maps[label][name].numpy(), cmap='inferno')
        axes[row, col].set_title(f'{name}: {label}')
        axes[row, col].axis('off')
plt.tight_layout()
plt.show()

## 7. Attention sink: where the attention mass goes in a language model

[Xiao et al. (2023)](https://arxiv.org/abs/2309.17453) noticed that in trained autoregressive language
models a large share of attention goes to the first token of the sequence, whatever its meaning.
Softmax must distribute one unit of mass, and it pays off for the model to put "spare" attention where
it does no harm. Let us check on GPT-2 small: for each layer we compute what share of attention the
tokens from position 1 onward give to the first token, averaged over the 12 heads and over positions.
For comparison, the same share under uniform attention, when the token at position $i$ splits its mass
among $i+1$ tokens.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained('gpt2')
lm = AutoModelForCausalLM.from_pretrained('gpt2', attn_implementation='eager').eval()

text = 'When Mary and John went to the store, John gave a drink to Mary.'
ids = tok(text, return_tensors='pt')
with torch.no_grad():
    att = lm(**ids, output_attentions=True).attentions      # 12 tensors (batch, heads, tokens, tokens)

n = ids['input_ids'].shape[1]
shares = torch.stack([a[0, :, 1:, 0].mean() for a in att])  # share of the first token, mean over heads and positions 1..n-1
uniform = float((1.0 / torch.arange(2, n + 1, dtype=torch.float32)).mean())
print(f'tokens: {n}, first token: {tok.convert_ids_to_tokens(ids["input_ids"][0, :1])}')
for layer, s in enumerate(shares, start=1):
    print(f'layer {layer:2}: {float(s):.3f}')
print(f'share under uniform attention: {uniform:.3f}')

**Task 3.** In which layer is the share of attention on the first token the largest? Take the attention
map of that layer averaged over heads, strike out the first token (its row and column), renormalize the
rows and find which token the last token of the phrase looks at most after that.

In [ ]:
# Your code here

## What to take away from the notebook

- **The raw attention map and rollout are class-blind**, and this is checked bit for bit: the
  correlation of the maps for two classes equals one. They show where information flows, not what
  influenced the answer.
- **The class enters the map only through the gradient.** Attention × gradient and Chefer differ from
  their blind twins by one factor, the gradient of the logit with respect to the attention matrix.
- **The maps of different methods diverge** even on one input and one model. Choose the method by the
  question, not by how pretty the map is.
- **In language models the first token takes most of the attention.** Subtract this column before
  reading an attention map.